# Paper Classification — Full pipeline 10 steps (Colab T4 GPU)

End-to-end pipeline per UPGRADE_ROADMAP_v2 Nhiệm vụ 1-5. Total time ≈ 5-8 hours on T4 free tier (within 12h session limit).

**Run order** (cells below execute in this exact sequence on `Run all`):

1. **Verify GPU + disk** (Guard #4: ≥20GB free)
2. **Clone repo + install deps**
3. **Mount Drive + restore cached artifacts** (Guard #3)
4. **Phase 0 — sanitize** (produces gold + main_2024 + inference_corpus parquets)
5. **Phase C M1 — TAPT** (continued MLM pretraining, ~30 min, skip if cached)
6. **Phase 1 — LLM augment** (8 classes: Special edu/ECE/TVET/LLL + 4 new Fields, ~$40 OpenAI, cache-aware)
7. **Phase 2 — train SPECTER2** (3 seeds × 3 tasks = 9 models, ~3-4h on T4)
8. **Phase D — kNN retrieval** (cache predictions for val + test, ~5 min)
9. **Phase A — GPT-5 panel** (cache predictions for val + test, ~$25 OpenAI, cache-aware)
10. **Phase 3 — evaluate** (M2 disabled; produces eval_report.json)
11. **Phase 4 — inference 2015-2024 corpus** (predictions_2015_2024.parquet)
12. **Phase 4b — export comprehensive review workbook** (27 sheets covering 2015-2024)
13. **Phase 5 — print_summary** (PRIMARY metric: high_support_macro_f1)
14. **Phase 5b — phase_summary** (side-by-side variant comparison)
15. **Download artifacts** (eval report + review.xlsx)

Encoder selection: `config.USE_TAPT=True` makes train/eval/inference auto-pick the TAPT-adapted encoder at `outputs/specter2_tapt/` when it exists.

**Rollback triggers** (run cell 16 at end for automated Gate check):
- Fields PRIMARY (high_support_macro_f1) on test 2024 < 0.55 → set `config.MULTILABEL_LOSS = 'bce_pos_weight'` + re-run cell 8 (train).

## 1. Verify GPU + disk (Guard #4)

In [ ]:
!nvidia-smi

In [ ]:
import subprocess
out = subprocess.check_output(['df', '-BG', '/content']).decode()
print(out)
avail = int(out.splitlines()[1].split()[3].rstrip('G'))
assert avail >= 20, f'Need at least 20GB free, got {avail}GB. Restart runtime.'
print(f'Disk OK: {avail}GB free')

## 2. Clone repo + install dependencies

In [ ]:
import os
if not os.path.exists('paper-classification'):
    !git clone https://github.com/harnetlinh/paper-classification.git
%cd paper-classification
!git pull
!git log --oneline -5

In [ ]:
!pip install -q pyarrow openpyxl python-dotenv tenacity openai==1.57.4 transformers==4.45.2

## 3. Mount Drive + restore cached artifacts

Drive path: `/content/drive/MyDrive/KHGDVN/`

Everything in `outputs/` is symlinked to Drive so every save touches Drive directly. After this cell, any previously cached artifact (sanitize parquets, TAPT model, LLM augment + classify parquets, kNN parquets, trained models, eval report) is restored. Cell-level idempotence means each step below will skip itself if its expected output is already present.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/KHGDVN'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/outputs', exist_ok=True)

import shutil
# Symlink outputs/ to Drive (one-time setup)
if os.path.exists('outputs') and not os.path.islink('outputs'):
    for item in os.listdir('outputs'):
        src = f'outputs/{item}'
        dst = f'{DRIVE_DIR}/outputs/{item}'
        if not os.path.exists(dst):
            shutil.move(src, dst)
    shutil.rmtree('outputs', ignore_errors=True)
if not os.path.islink('outputs'):
    os.symlink(f'{DRIVE_DIR}/outputs', 'outputs')

# Data + .env symlinks
os.makedirs('data', exist_ok=True)
data_xlsx = '2025_11_09_-_biblio_-_du_lieu_phep_phan_tich_-_2013-2024__1_.xlsx'
drive_data = f'{DRIVE_DIR}/data/{data_xlsx}'
if os.path.exists(drive_data) and not os.path.exists(f'data/{data_xlsx}'):
    os.symlink(drive_data, f'data/{data_xlsx}')
if os.path.exists(f'{DRIVE_DIR}/.env') and not os.path.exists('.env'):
    shutil.copy(f'{DRIVE_DIR}/.env', '.env')

!ls -la outputs/ 2>/dev/null | head -25
!ls -la data/ 2>/dev/null

In [ ]:
# First-time setup: upload Excel + .env if not on Drive yet
from google.colab import files
if not os.path.exists(f'data/{data_xlsx}'):
    print('Upload the Excel data file:')
    uploaded = files.upload()
    for fn in uploaded.keys():
        os.rename(fn, f'data/{fn}')
        os.makedirs(f'{DRIVE_DIR}/data', exist_ok=True)
        shutil.copy(f'data/{fn}', f'{DRIVE_DIR}/data/{fn}')
if not os.path.exists('.env'):
    print('Paste .env content (OPENAI_API_KEY=sk-...). Empty line + Ctrl+D to finish:')
    env = ''
    try:
        while True:
            env += input() + '\n'
    except EOFError:
        pass
    open('.env', 'w').write(env)
    shutil.copy('.env', f'{DRIVE_DIR}/.env')
print('Data + .env ready.')

## Step 4 — Phase 0: sanitize

Produces `gold_2013_2023.parquet`, `main_2024_clean.parquet`, and `inference_corpus_2015_2024.parquet` (NHIỆM VỤ 5: hybrid labeled + unlabeled 2015-2024 corpus for inference).

In [ ]:
if not os.path.exists('outputs/inference_corpus_2015_2024.parquet'):
    !python sanitize.py
else:
    print('Sanitize parquets cached. Skipping.')
!ls outputs/*.parquet 2>/dev/null

## Step 5 — Phase C M1: TAPT (continued MLM pretraining)

Adapts SPECTER2 base to the 2015-2024 corpus vocabulary via masked-LM pretraining (3 epochs, ~30 min on T4). Skipped if cached. The trained encoder is saved at `outputs/specter2_tapt/` — train/eval/inference auto-load it when `config.USE_TAPT=True`.

In [ ]:
if not os.path.exists('outputs/specter2_tapt/config.json'):
    !python tapt.py --epochs 3 --lr 5e-5 --batch-size 16
else:
    print('TAPT artifact cached. Skipping ~30min step.')
!ls outputs/specter2_tapt/ 2>/dev/null

## Step 6 — Phase 1: LLM augment 8 classes

Augments rare/underperforming classes via GPT-5 panel vote (NHIỆM VỤ 4 extension):
- **4 rare classes** (unanimous 3/3): Special education, ECE, TVET, LLL
- **4 underperforming Fields** (majority 2/3): test and assessment, curriculum, Non-STEM Education, Education economically

Per-prompt cache → re-runs are FREE. Cost first run ~$40-50 OpenAI for all 8 classes.

In [ ]:
!python llm_augment.py --class all

## Step 7 — Phase 2: train SPECTER2 ensemble (3 seeds × 3 tasks = 9 models)

AsymmetricLoss (Ridnik 2021) for Fields/Levels per NHIỆM VỤ 3. Focal CE for Method. Per-class threshold tuning with 3-tier support-aware grid + precision floor per NHIỆM VỤ 2.

Time: ~3-4h on T4. To force fresh re-train (e.g. after pulling new code), delete `outputs/model_*.pt` first.

In [ ]:
# Uncomment to force fresh retrain:
# !rm -f outputs/model_*.pt outputs/thresholds_*.json outputs/training_log_*.json
!python train_specter2.py --task all --ensemble

## Step 8 — Phase D: kNN retrieval ensemble member

SPECTER2 [CLS] embedding → top-10 neighbor soft-label vote. Cached per (split × task).

In [ ]:
!python knn_retrieval.py --split both --task all

## Step 9 — Phase A: GPT-5 panel ensemble member

3-model OpenAI panel full classification on val 2023 + test 2024. Cached per prompt — re-runs are FREE. Cost first run ~$25.

In [ ]:
!python llm_classify.py --split both

## Step 10 — Phase 3: evaluate

Per NHIỆM VỤ 1, `USE_QUANTIFICATION_AT_TEST=False` (M2 disabled — regressed -0.16 on Fields macro_F1).

Produces `eval_report.json` with: SPECTER2 baseline + Phase A (GPT-5 ensemble) + Phase D (kNN ensemble) + 3-way (SPECTER2 + GPT-5 + kNN), each with full per-class breakdown.

In [ ]:
!python evaluate.py --task all

## Step 11 — Phase 4: inference on 2015-2024 corpus (NHIỆM VỤ 5)

Predicts all 3 tasks on the hybrid corpus (~2900 papers: gold 2015-2023 + main 2024 + main 2015-2023 unlabeled). Produces `predictions_2015_2024.parquet` consumed by the next cell.

In [ ]:
!python inference.py --corpus outputs/inference_corpus_2015_2024.parquet

## Step 12 — Phase 4b: export comprehensive review workbook (NHIỆM VỤ 5)

When `predictions_2015_2024.parquet` exists (produced by Step 11), `export_review.py` auto-runs in NV5 mode and produces a 27-sheet workbook at `outputs/review_full_2015_2024.xlsx`:

| Group | Sheets | Purpose |
|---|---|---|
| Summary (4) | summary_all, summary_val_2023, summary_test_2024, summary_inference_only | One row per paper |
| Disagreements (4) | {,major_}disagreements_{val,test} | Sorted human-machine disagreement views |
| Per-class (9) | fields/levels/method × {val, test, inference_only} | Per-class human/pred/prob/status |
| Stats per-split (6) | stats_{val,test}_{fields,levels,method} | precision/recall/F1 |
| Stats per-year (3) | stats_per_year_{fields,levels,method} | Row=class, Col=year |
| Legend (1) | legend | Column glossary |

In [ ]:
!python export_review.py

## Step 13 — Phase 5: print_summary (PRIMARY publication metric)

Per NHIỆM VỤ 1, the headline metric is `high_support_macro_f1` — macro F1 over classes with val_support ≥ {fields:50, levels:30, method:20}. Raw `macro_f1` reported as secondary.

In [ ]:
!python print_summary.py

## Step 14 — Phase 5b: phase_summary (variant side-by-side)

Compares SPECTER2 baseline vs +GPT-5 panel vs +kNN vs full 3-way ensemble. Per-class breakdown for each task.

In [ ]:
!python phase_summary.py

## Step 15 — Automated rollback gate check

Reads `outputs/eval_report.json` + per-task PRIMARY thresholds. If Fields PRIMARY < 0.55 on test_2024, signals AsymmetricLoss regression — rollback to BCE+pos_weight.

In [ ]:
import json, numpy as np
import config as _cfg
with open('outputs/eval_report.json') as f:
    rep = json.load(f)

def primary_test(task):
    r = rep.get('tasks', {}).get(task, {})
    table = r.get('per_class_table', [])
    thr = _cfg.HIGH_SUPPORT_VAL_THRESHOLDS.get(task, 30)
    idx = [i for i, e in enumerate(table) if e.get('val_support', 0) >= thr]
    if not idx:
        return float('nan'), 0
    f1s = [table[i].get('test_f1', 0.0) for i in idx]
    return float(np.mean(f1s)), len(idx)

print('PRIMARY metric (high_support_macro_f1) on test 2024:')
rollback_needed = False
for task in ['fields', 'levels', 'method']:
    f1, n = primary_test(task)
    threshold = 0.55
    flag = ' ⚠ ROLLBACK' if (task in ('fields', 'levels') and f1 < threshold) else ''
    if task in ('fields', 'levels') and f1 < threshold:
        rollback_needed = True
    print(f'  {task:<8} ({n} classes): {f1:.4f}{flag}')

if rollback_needed:
    print()
    print('ROLLBACK INSTRUCTIONS (per UPGRADE_ROADMAP_v2):')
    print('  1. Edit config.py: MULTILABEL_LOSS = "bce_pos_weight"')
    print('  2. Delete outputs/model_*.pt + outputs/thresholds_*.json + outputs/training_log_*.json')
    print('  3. Re-run cell 8 (Phase 2 train) + cell 11 (evaluate) + this cell')
else:
    print()
    print('Gate PASS: keep AsymmetricLoss + current config.')

## Step 16 — Download artifacts

Heavy artifacts (TAPT model ~440MB, 9 model checkpoints) stay on Drive. This zip contains just the reports + review workbook for local archival.

In [ ]:
!zip -r artifacts.zip \
    outputs/eval_report.json \
    outputs/training_log_*.json \
    outputs/thresholds_*.json \
    outputs/review_full_2015_2024.xlsx \
    outputs/predictions_2015_2024.parquet 2>/dev/null
from google.colab import files
files.download('artifacts.zip')

## Resume after disconnect

Every cell is **idempotent** — re-running after disconnect skips already-completed steps. Heavy artifacts live on Drive via the `outputs/` symlink, so:

1. Re-run cells 1-4 (GPU, deps, mount Drive).
2. Subsequent cells skip themselves when their sentinel output exists:
   - Step 4 (sanitize) skips if `outputs/inference_corpus_2015_2024.parquet` exists
   - Step 5 (TAPT) skips if `outputs/specter2_tapt/config.json` exists
   - Step 6 (LLM augment) — per-prompt cache, re-runs free
   - Step 9 (LLM classify) — per-prompt cache, re-runs free
   - Step 7 (train) — DOES re-run (no skip). To preserve cached models, comment out the train cell OR delete first.

Manual force-rerun:
```bash
!rm -rf outputs/specter2_tapt/                 # force TAPT
!rm -f outputs/model_*.pt outputs/thresholds_*.json outputs/training_log_*.json  # force re-train
!rm -rf outputs/llm_classify/ outputs/llm_logs/  # force GPT-5 re-call (costs $25+)
!rm -rf outputs/knn_retrieval/                 # force kNN re-encode
!rm outputs/inference_corpus_2015_2024.parquet outputs/predictions_2015_2024.parquet  # force re-sanitize + re-inference
```